In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, SimpleRNN

2026-01-13 06:13:42.637256: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
## Loading the dataset
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

## Printing the first review and its label
print("First review (as word indices):\n", X_train[0], "\n\nLength:", len(X_train[0]))
print("\nLabel of the first review:", y_train[0])

First review (as word indices):
 [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32] 

Length: 218

Label of the

In [3]:
## Shape of the training and testing data
print("Shape of the Training set:", (X_train.shape, y_train.shape))
print("Shape of the Testing set:", (X_test.shape, y_test.shape))

Shape of the Training set: ((25000,), (25000,))
Shape of the Testing set: ((25000,), (25000,))


In [4]:
## Checking a review
sample_review = X_train[0]
sample_review_label = y_train[0]

print(f"Sample review (word indices):\n{sample_review}\n")
print(f"Sample review label: {sample_review_label}")

Sample review (word indices):
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]

Sample review label: 1


In [5]:
## Feeding all the index to a variable
word_idx = imdb.get_word_index()
word_idx

{'fawn': 34701,
 'tsukino': 52006,
 'nunnery': 52007,
 'sonja': 16816,
 'vani': 63951,
 'woods': 1408,
 'spiders': 16115,
 'hanging': 2345,
 'woody': 2289,
 'trawling': 52008,
 "hold's": 52009,
 'comically': 11307,
 'localized': 40830,
 'disobeying': 30568,
 "'royale": 52010,
 "harpo's": 40831,
 'canet': 52011,
 'aileen': 19313,
 'acurately': 52012,
 "diplomat's": 52013,
 'rickman': 25242,
 'arranged': 6746,
 'rumbustious': 52014,
 'familiarness': 52015,
 "spider'": 52016,
 'hahahah': 68804,
 "wood'": 52017,
 'transvestism': 40833,
 "hangin'": 34702,
 'bringing': 2338,
 'seamier': 40834,
 'wooded': 34703,
 'bravora': 52018,
 'grueling': 16817,
 'wooden': 1636,
 'wednesday': 16818,
 "'prix": 52019,
 'altagracia': 34704,
 'circuitry': 52020,
 'crotch': 11585,
 'busybody': 57766,
 "tart'n'tangy": 52021,
 'burgade': 14129,
 'thrace': 52023,
 "tom's": 11038,
 'snuggles': 52025,
 'francesco': 29114,
 'complainers': 52027,
 'templarios': 52125,
 '272': 40835,
 '273': 52028,
 'zaniacs': 52130,

In [6]:
## Checking sample vector and converting back to words
sentence = {value: key for key, value in word_idx.items()}
decoded_review = ' '.join([sentence.get(i - 3, '') for i in sample_review])
print("Sample review encoded:\n", sample_review)
print("Decoded review:\n", decoded_review)

Sample review encoded:
 [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Decoded review:
  this film was just

In [7]:
for i in range(len(sample_review)):
    print(f"i: {i}, i-3:{i-3}")
    print(f"Index: {sample_review[i]}  Word: {sentence.get(sample_review[i] - 3, '')}")
    print("-----")

i: 0, i-3:-3
Index: 1  Word: 
-----
i: 1, i-3:-2
Index: 14  Word: this
-----
i: 2, i-3:-1
Index: 22  Word: film
-----
i: 3, i-3:0
Index: 16  Word: was
-----
i: 4, i-3:1
Index: 43  Word: just
-----
i: 5, i-3:2
Index: 530  Word: brilliant
-----
i: 6, i-3:3
Index: 973  Word: casting
-----
i: 7, i-3:4
Index: 1622  Word: location
-----
i: 8, i-3:5
Index: 1385  Word: scenery
-----
i: 9, i-3:6
Index: 65  Word: story
-----
i: 10, i-3:7
Index: 458  Word: direction
-----
i: 11, i-3:8
Index: 4468  Word: everyone's
-----
i: 12, i-3:9
Index: 66  Word: really
-----
i: 13, i-3:10
Index: 3941  Word: suited
-----
i: 14, i-3:11
Index: 4  Word: the
-----
i: 15, i-3:12
Index: 173  Word: part
-----
i: 16, i-3:13
Index: 36  Word: they
-----
i: 17, i-3:14
Index: 256  Word: played
-----
i: 18, i-3:15
Index: 5  Word: and
-----
i: 19, i-3:16
Index: 25  Word: you
-----
i: 20, i-3:17
Index: 100  Word: could
-----
i: 21, i-3:18
Index: 43  Word: just
-----
i: 22, i-3:19
Index: 838  Word: imagine
-----
i: 23, i-3:20

In [8]:
## setting the max length of the vectors
sent_len = 500

In [9]:
## Applying pad_sequence
X_train = sequence.pad_sequences(X_train, maxlen=sent_len)
X_test = sequence.pad_sequences(X_test, maxlen=sent_len)

In [10]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

gpus = tf.config.list_physical_devices('GPU')
display(tf.config.experimental.get_memory_info('GPU:0'))
# tf.config.optimizer.set_jit(True)
tf.keras.backend.clear_session()

I0000 00:00:1768265026.906598   12974 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10320 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:29:00.0, compute capability: 7.5


{'current': 0, 'peak': 0}

In [11]:
## Defining early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

# Train Simple RNN
model = Sequential()
model.add(Embedding(vocab_size,128, input_length = sent_len))
model.add(SimpleRNN(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1,activation='sigmoid'))

/home/prashant/.conda/envs/deeplearning/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [14]:
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size = 32,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/25


2026-01-13 06:13:48.081200: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d6c78003030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-13 06:13:48.081213: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 2060, Compute Capability 7.5
2026-01-13 06:13:48.103658: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-13 06:13:48.241199: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


  5/625 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.4497 - loss: 0.6932

I0000 00:00:1768265028.846591   13053 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 39ms/step - accuracy: 0.6347 - loss: 35.3163 - val_accuracy: 0.4966 - val_loss: 0.7332
Epoch 2/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.7063 - loss: 34.9409 - val_accuracy: 0.7240 - val_loss: 0.5458
Epoch 3/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 41ms/step - accuracy: 0.7401 - loss: 1322.8708 - val_accuracy: 0.6690 - val_loss: 0.6002
Epoch 4/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.8088 - loss: 0.4412 - val_accuracy: 0.7852 - val_loss: 0.4678
Epoch 5/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.8834 - loss: 0.2843 - val_accuracy: 0.8174 - val_loss: 0.4146
Epoch 6/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.9273 - loss: 0.1940 - val_accuracy: 0.8206 - val_loss: 0.4493
Epoch 7/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9540 - loss: 0.1311 - val_accuracy: 0.8134 - val_loss: 0.5050
Epoch 8/25
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9679 - loss: 0.0972 - val_ac

In [16]:
model.save('rnn_imdb_ac9901.h5')